# NB08 — Agentic NIAH Stress Test (Exploratory Analysis)
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

## What this notebook does

Adapts the **Needle-in-a-Haystack (NIAH)** benchmark ([Kamradt, 2023](https://github.com/gkamradt/LLMTest_NeedleInAHaystack)) to stress-test the **agentic RAG pipeline** specifically.

The standard NIAH grid is (depth × context_length). The agentic pipeline adds two extra dimensions not testable with static RAG:

| Dimension | Standard RAG | Agentic RAG |
|-----------|-------------|-------------|
| `needle_retrieved` | ✓ | ✓ |
| `needle_in_answer` | ✓ | ✓ |
| `subquery_targets_needle` | — | ✓ **new** |
| `iteration_retrieved` | — | ✓ **new** |
| `n_iterations` | — | ✓ **new** |

## Research questions answered

1. **Depth × context recovery** — Does multi-pass retrieval rescue needles that single-pass misses?
2. **Convergence speed** — Does the agent converge faster (fewer iterations) when the needle is shallow vs. deep?
3. **Sub-query quality** — Does the agent's query decomposition spontaneously generate sub-queries targeting the needle topic?
4. **Agentic advantage** — Which (depth, context_length) cells does agentic outperform standard, and vice versa?

## Outputs
- `reports/niah_results_agentic_rag.json`
- `reports/figures/fig_niah_agentic_heatmap.png`
- `reports/figures/fig_niah_iteration_convergence.png`
- `reports/figures/fig_niah_subquery_quality.png`
- `reports/figures/fig_niah_cross_method_comparison.png`
- `reports/tables/niah_cross_method_comparison.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.io import load_env, project_root, data_dir
load_env()

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

FIG_DIR = project_root() / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
print('Setup complete.')

---
## 1. Initialise the agent

Re-uses the shared Chroma vector store built in NB01.

In [ ]:
from src.methods.standard_rag.langchain_pipeline import load_vector_store, build_rag_chain
from src.methods.agentic_rag.agent import AgenticRAGAgent

vs = load_vector_store()
_, langchain_retriever = build_rag_chain(vs)

# Adapter so AgenticRAGAgent can call .retrieve() on the LangChain retriever
class LangChainRetrieverAdapter:
    def __init__(self, retriever, chroma_store):
        self.retriever = retriever
        self._chroma = chroma_store   # exposed for niah.py to inject/remove docs

    def retrieve(self, query: str, top_k: int = 10) -> pd.DataFrame:
        docs = self.retriever.invoke(query)[:top_k]
        return pd.DataFrame([{
            'doc_id': d.metadata.get('source', ''),
            'text':   d.page_content,
        } for d in docs])

adapter = LangChainRetrieverAdapter(langchain_retriever, vs)
agent = AgenticRAGAgent(vector_pipeline=adapter)
print('Agent ready.')

---
## 2. Configure the NIAH sweep

Grid identical to the standard RAG NIAH for direct comparability.

In [ ]:
from src.methods.agentic_rag.niah import (
    AgenticNIAHTester,
    DEFAULT_NEEDLE, DEFAULT_QUESTION, DEFAULT_EXPECTED,
)

# Mirror the grid from langchain_pipeline.py
CONTEXT_LENGTHS = [2_000, 8_000, 16_000, 32_000]   # tokens (approx)
DEPTHS          = [0.1, 0.25, 0.5, 0.75, 0.9]       # fraction through haystack

tester = AgenticNIAHTester(
    agent=agent,
    context_lengths=CONTEXT_LENGTHS,
    depths=DEPTHS,
    needle=DEFAULT_NEEDLE,
    question=DEFAULT_QUESTION,
    expected=DEFAULT_EXPECTED,
    use_deepeval=True,
)

print(f'Grid: {len(CONTEXT_LENGTHS)} context lengths × {len(DEPTHS)} depths = {len(CONTEXT_LENGTHS)*len(DEPTHS)} cells')
print(f'Needle: "{DEFAULT_NEEDLE[:80]}..."')

---
## 3. Run full sweep

⚠️ **Cost estimate**: ~20 cells × (decomposition + 2 retrieval iterations + synthesis + DeepEval faithfulness) ≈ 80–120 GPT-4o calls. Typical runtime: 8–15 min.

To run a faster smoke test, set `CONTEXT_LENGTHS = [2_000, 8_000]` and `DEPTHS = [0.1, 0.5, 0.9]` above.

In [ ]:
RESULTS_PATH = project_root() / 'reports' / 'niah_results_agentic_rag.json'

if RESULTS_PATH.exists():
    print(f'Loading cached results from {RESULTS_PATH}')
    raw = AgenticNIAHTester.load(RESULTS_PATH)
    # Reconstruct NIAHCellResult objects for downstream plotting
    from src.methods.agentic_rag.niah import NIAHCellResult
    results = {
        k: NIAHCellResult(
            context_length=v['context_length'],
            depth=v['depth'],
            subquery_targets_needle=v['subquery_targets_needle'],
            needle_retrieved=v['needle_retrieved'],
            needle_in_synthesis=v['needle_in_synthesis'],
            iteration_retrieved=v['iteration_retrieved'],
            n_iterations=v['n_iterations'],
            faithfulness_score=v['faithfulness_score'],
            latency_s=v['latency_s'],
            answer_snippet=v['answer_snippet'],
            sub_queries=v.get('sub_queries', []),
        )
        for k, v in raw.items()
    }
else:
    results = tester.run_sweep()
    AgenticNIAHTester.save(results, RESULTS_PATH)

print(f'Loaded {len(results)} cells.')
passed = sum(1 for r in results.values() if r.passed())
print(f'Overall pass rate: {passed}/{len(results)} ({100*passed/len(results):.0f}%)')

---
## 4. Flatten results to DataFrame

In [ ]:
df = pd.DataFrame([r.to_dict() for r in results.values()])
df = df.sort_values(['context_length', 'depth']).reset_index(drop=True)
display(df[['context_length', 'depth', 'subquery_targets_needle',
            'needle_retrieved', 'needle_in_synthesis', 'passed',
            'iteration_retrieved', 'n_iterations', 'latency_s', 'faithfulness_score']])

---
## 5. Fig A — Pass/Fail heatmap (depth × context length)

Direct analogue of the standard RAG NIAH heatmap.

In [ ]:
pivot_pass = df.pivot(index='context_length', columns='depth', values='passed').astype(float)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    pivot_pass,
    annot=pivot_pass.applymap(lambda x: '✓' if x == 1 else '✗'),
    fmt='',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Pass (1) / Fail (0)'},
    ax=ax,
)
ax.set_xlabel('Depth (fraction through haystack)')
ax.set_ylabel('Context length (approx. tokens)')
ax.set_title('Agentic NIAH — Pass/Fail Grid')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_niah_agentic_heatmap.pdf', dpi=300)
fig.savefig(FIG_DIR / 'fig_niah_agentic_heatmap.png', dpi=300)
plt.show()

---
## 6. Fig B — Iteration convergence heatmap

Which iteration (1, 2, 3 ...) first retrieved the needle?
White cells = needle never retrieved.

In [ ]:
pivot_iter = df.pivot(index='context_length', columns='depth', values='iteration_retrieved')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    pivot_iter,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd_r',
    linewidths=0.5,
    cbar_kws={'label': 'Iteration at first retrieval (lower = faster)'},
    ax=ax,
    mask=pivot_iter.isna(),
)
ax.set_xlabel('Depth')
ax.set_ylabel('Context length')
ax.set_title('Agentic NIAH — Iteration at First Needle Retrieval')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_niah_iteration_convergence.pdf', dpi=300)
fig.savefig(FIG_DIR / 'fig_niah_iteration_convergence.png', dpi=300)
plt.show()

---
## 7. Fig C — Sub-query quality analysis

Did the agent's query decomposition spontaneously generate a sub-query targeting the needle?
Overlay on the pass/fail grid to show whether sub-query targeting was *necessary* for success.

In [ ]:
pivot_subq = df.pivot(
    index='context_length', columns='depth', values='subquery_targets_needle'
).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, data, title, fmt in [
    (axes[0], pivot_subq, 'Sub-query targets needle', ''),
    (axes[1], pivot_pass,  'Pass/Fail',               ''),
]:
    sns.heatmap(
        data,
        annot=data.applymap(lambda x: '✓' if x == 1 else '✗'),
        fmt=fmt,
        cmap='RdYlGn',
        vmin=0, vmax=1,
        linewidths=0.5,
        cbar=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Depth')
    ax.set_ylabel('Context length')

fig.suptitle('Sub-query Targeting vs. Pass/Fail (Agentic NIAH)', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_niah_subquery_quality.pdf', dpi=300)
fig.savefig(FIG_DIR / 'fig_niah_subquery_quality.png', dpi=300)
plt.show()

# Contingency: did sub-query targeting predict success?
ct = pd.crosstab(df['subquery_targets_needle'], df['passed'],
                 rownames=['subquery_hit'], colnames=['passed'])
print('Contingency table — sub-query targeting vs. pass:')
print(ct)

---
## 8. Fig D — Cross-method comparison: Agentic vs Standard RAG

Requires `reports/niah_results_standard_rag.json` (produced by NB03).

In [ ]:
std_path = project_root() / 'reports' / 'niah_results_standard_rag.json'

if std_path.exists():
    comp_df = AgenticNIAHTester.compare_with_standard(results, std_path)
    comp_df.to_csv(project_root() / 'reports' / 'tables' / 'niah_cross_method_comparison.csv', index=False)

    # --- Side-by-side heatmaps ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    for ax, col, title, cmap in [
        (axes[0], 'std_passed',  'Standard RAG',  'RdYlGn'),
        (axes[1], 'agt_passed',  'Agentic RAG',   'RdYlGn'),
        (axes[2], 'agentic_advantage', 'Agentic advantage\n(agentic✓ & std✗)', 'Blues'),
    ]:
        pivot = comp_df.pivot(index='context_length', columns='depth', values=col).astype(float)
        sns.heatmap(
            pivot,
            annot=pivot.applymap(lambda x: '✓' if x == 1 else ('▲' if col == 'agentic_advantage' and x == 1 else '✗')),
            fmt='',
            cmap=cmap,
            vmin=0, vmax=1,
            linewidths=0.5,
            cbar=False,
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel('Depth')
        ax.set_ylabel('Context length')

    fig.suptitle('NIAH Cross-Method Comparison', y=1.02, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'fig_niah_cross_method_comparison.pdf', dpi=300)
    fig.savefig(FIG_DIR / 'fig_niah_cross_method_comparison.png', dpi=300)
    plt.show()

    # --- Latency comparison ---
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    comp_df.plot.scatter(
        x='std_latency', y='agt_latency',
        c='agentic_advantage', colormap='RdYlGn',
        s=60, alpha=0.8, ax=ax2
    )
    lim = max(comp_df[['std_latency', 'agt_latency']].max().max(), 1)
    ax2.plot([0, lim], [0, lim], 'k--', lw=0.8, label='Equal latency')
    ax2.set_xlabel('Standard RAG latency (s)')
    ax2.set_ylabel('Agentic RAG latency (s)')
    ax2.set_title('Latency: Standard vs Agentic RAG per NIAH cell')
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(FIG_DIR / 'fig_niah_latency_comparison.png', dpi=300)
    plt.show()

    print('\n── Cross-method summary ──')
    print(f"Standard RAG pass rate:  {comp_df['std_passed'].mean():.1%}")
    print(f"Agentic RAG pass rate:   {comp_df['agt_passed'].mean():.1%}")
    print(f"Cells where agentic wins: {comp_df['agentic_advantage'].sum()}")
    print(f"Cells where standard wins: {comp_df['standard_advantage'].sum()}")
    display(comp_df)
else:
    print('Standard RAG NIAH results not found — run NB03 first.')
    print(f'Expected at: {std_path}')

---
## 9. Faithfulness score comparison

In [ ]:
faith_df = df.dropna(subset=['faithfulness_score'])

if len(faith_df) > 0:
    pivot_faith = faith_df.pivot(index='context_length', columns='depth', values='faithfulness_score')

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(
        pivot_faith,
        annot=True, fmt='.2f',
        cmap='YlGn', vmin=0, vmax=1,
        linewidths=0.5,
        cbar_kws={'label': 'DeepEval Faithfulness Score'},
        ax=ax,
    )
    ax.set_xlabel('Depth')
    ax.set_ylabel('Context length')
    ax.set_title('Agentic NIAH — DeepEval Faithfulness per Cell')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'fig_niah_faithfulness.png', dpi=300)
    plt.show()

    print(f'Mean faithfulness: {faith_df["faithfulness_score"].mean():.3f}')
    print(f'Min faithfulness:  {faith_df["faithfulness_score"].min():.3f} '
          f'(ctx={faith_df.loc[faith_df["faithfulness_score"].idxmin(), "context_length"]}, '
          f'depth={faith_df.loc[faith_df["faithfulness_score"].idxmin(), "depth"]})')
else:
    print('No faithfulness scores available — set use_deepeval=True in tester config.')

---
## 10. Summary statistics table

In [ ]:
summary = pd.DataFrame({
    'metric': [
        'Total cells',
        'Pass rate (retrieved + in synthesis)',
        'Needle retrieved rate',
        'Needle in synthesis rate',
        'Sub-query targeting rate',
        'Mean iterations (all cells)',
        'Mean iterations (pass only)',
        'Mean iteration at first retrieval (pass cells)',
        'Mean latency (s)',
        'Mean DeepEval faithfulness',
    ],
    'value': [
        len(df),
        f"{df['passed'].mean():.1%}",
        f"{df['needle_retrieved'].mean():.1%}",
        f"{df['needle_in_synthesis'].mean():.1%}",
        f"{df['subquery_targets_needle'].mean():.1%}",
        f"{df['n_iterations'].mean():.1f}",
        f"{df[df['passed']]['n_iterations'].mean():.1f}" if df['passed'].any() else 'n/a',
        f"{df[df['passed']]['iteration_retrieved'].mean():.1f}" if df['passed'].any() else 'n/a',
        f"{df['latency_s'].mean():.1f}s",
        f"{df['faithfulness_score'].mean():.3f}" if df['faithfulness_score'].notna().any() else 'n/a',
    ],
})
display(summary)
summary.to_csv(project_root() / 'reports' / 'tables' / 'niah_agentic_summary.csv', index=False)
print('Summary saved.')

---
## Notes for manuscript

- **Interpretation guide**: A cell where `subquery_targets_needle=True` but `needle_retrieved=False` suggests the agent's sub-query semantics are aligned but the vector store failed to surface the chunk — a retrieval quality issue, not a planning issue.
- **Iteration convergence**: Cells where `iteration_retrieved > 1` show that agentic refinement rescued retrievals that failed on the first pass — the core advantage over standard RAG.
- **Agentic disadvantage cells**: Higher latency is expected — the meaningful comparison is whether the accuracy gain justifies the additional cost per query.
- Reference: Kamradt G. *Needle In A Haystack — Pressure Testing LLMs.* 2023. https://github.com/gkamradt/LLMTest_NeedleInAHaystack